In [1]:
import numpy as np
import os
import requests
import tarfile
#import cv2
from glob import glob
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.optimizers.schedules import LearningRateSchedule
from tensorflow.keras import backend as K
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.preprocessing import image
from keras.utils import plot_model 

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from pydensecrf import densecrf
from pydensecrf.utils import unary_from_softmax

ModuleNotFoundError: No module named 'pydensecrf'

In [ ]:
## Pré-traitement des images : Correspondance des classes avec la colormap,
## redimensionnement, normalisation standard, augmentaiton de données

# Définition de la taille souhaitée pour les images, i.e. taille d'entrée du backbone resnet101
target_size = (224, 224)
# Correspondance classe -> colormap
voc_classes = {
    'background': 0,
    'aeroplane': 1,
    'bicycle': 2,
    'bird': 3,
    'boat': 4,
    'bottle': 5,
    'bus': 6,
    'car': 7,
    'cat': 8,
    'chair': 9,
    'cow': 10,
    'diningtable': 11,
    'dog': 12,
    'horse': 13,
    'motorbike': 14,
    'person': 15,
    'pottedplant': 16,
    'sheep': 17,
    'sofa': 18,
    'train': 19,
    'tvmonitor': 20
}
VOC_COLORMAP = [
    [0, 0, 0],
    [128, 0, 0],
    [0, 128, 0],
    [128, 128, 0],
    [0, 0, 128],
    [128, 0, 128],
    [0, 128, 128],
    [128, 128, 128],
    [64, 0, 0],
    [192, 0, 0],
    [64, 128, 0],
    [192, 128, 0],
    [64, 0, 128],
    [192, 0, 128],
    [64, 128, 128],
    [192, 128, 128],
    [0, 64, 0],
    [128, 64, 0],
    [0, 192, 0],
    [128, 192, 0],
    [0, 64, 128],
]

def load_and_preprocess_image(image_path, annotation_path):
    # Charger l'image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)

    # Charger l'image d'annotation
    annotation = tf.io.read_file(annotation_path)
    annotation = tf.image.decode_png(annotation, channels=3)

    # Convertir l'image en float32 et normaliser
    image = tf.image.convert_image_dtype(image, tf.float32) / 255.0

    # Redimensionner l'image
    image = tf.image.resize(image, target_size)

    # Redimensionner l'annotation et convertir en tenseur avec les numéros de classe
    annotation = tf.image.resize(annotation, target_size)
    annotation = tf.py_function(func=convert_to_class_tensor, inp=[annotation], Tout=tf.uint8)
    annotation.set_shape([target_size[0], target_size[1], 1])
    
    return image, annotation

def convert_to_class_tensor(annotation):
    # Fonction pour convertir l'annotation en tenseur de classe
    annotation = tf.py_function(func=map_color_to_class, inp=[annotation], Tout=tf.uint8)
    return annotation

def map_color_to_class(annotation):
    # Fonction pour mapper les couleurs aux classes
    annotation_tensor = tf.zeros_like(annotation[..., 0], dtype=tf.uint8)
    for i, color in enumerate(VOC_COLORMAP):
        mask = tf.reduce_all(tf.equal(annotation, color), axis=-1)
        annotation_tensor = tf.where(mask, i, annotation_tensor)
    return tf.expand_dims(annotation_tensor, axis=-1)

def augment_data(image, annotation):
    image = tf.image.flip_left_right(image)
    image = tf.image.flip_up_down(image)

    annotation = tf.image.flip_left_right(annotation)
    annotation = tf.image.flip_up_down(annotation)

    return image, annotation

In [ ]:
## Séparation des données en jeu d'entrainement, de validation et de test
# Définition des chemins d'accès aux données
image_folder = "data/images/"
annotation_folder = "data/annotations/"
# Liste des fichiers d'annotations
annotation_files = glob(os.path.join(annotation_folder, "*.png"))
# Triez la liste des fichiers d'annotations
annotation_files = sorted(annotation_files)
# Création d'une liste de paires image/annotation
file_paths = []
# Parcourir les fichiers d'annotations et créer les paires correspondantes
for annotation_path in annotation_files:
    # Extraire le nom de fichier sans extension
    annotation_name = Path(annotation_path).stem
    # Construire le chemin de l'image correspondante
    image_path = os.path.join(image_folder, f"{annotation_name}.jpg")
    # Vérifier si l'image existe
    if os.path.exists(image_path):
        # Ajouter la paire à la liste
        file_paths.append((image_path, annotation_path))
# Séparation en ensembles d'entraînement et de test (85%/15%)
train_paths, test_paths = train_test_split(file_paths, test_size=0.15, random_state=42)
# Séparation de train_paths en ensembles d'entraînement et de validation (90%/10%)
train_paths, val_paths = train_test_split(train_paths, test_size=0.15, random_state=42)

In [ ]:
##Création des Datasets pour l'entraînement et le test
train_dataset = tf.data.Dataset.from_tensor_slices(train_paths)
train_dataset = train_dataset.map(lambda x: load_and_preprocess_image(x[0], x[1]))

val_dataset = tf.data.Dataset.from_tensor_slices(val_paths)
val_dataset = val_dataset.map(lambda x: load_and_preprocess_image(x[0], x[1]))

test_dataset = tf.data.Dataset.from_tensor_slices(test_paths)
test_dataset = test_dataset.map(lambda x: load_and_preprocess_image(x[0], x[1]))

# Prendre 20% du jeu de données initial
train_dataset_subset = train_dataset.take(int(0.2 * len(train_paths)))

# Appliquer la data augmentation sur le sous-ensemble
train_dataset_augmented = train_dataset_subset.map(augment_data, num_parallel_calls=tf.data.AUTOTUNE)

# Concaténer les données augmentées avec le jeu de données d'entraînement initial
train_dataset_combined = train_dataset.concatenate(train_dataset_augmented)

In [ ]:
## Création du modèle DeepLabV3

# Définir la taille d'entrée
input_shape = (224, 224, 3)  # Ajuster selon vos besoins

def Conv1x1(inputs, filters=256, dropout_rate=0.5):
        conv1x1 = layers.Conv2D(filters, (1, 1), padding='same')(inputs)
        conv1x1 = layers.BatchNormalization()(conv1x1)
        conv1x1 = layers.Activation('relu')(conv1x1)
        conv1x1 = layers.Dropout(dropout_rate)(conv1x1)
        return conv1x1

# Bloc ASPP
def aspp(inputs, filters=256, atrous_rates=[6, 12, 18], dropout_rate=0.5):
    atrous_blocks = []

    # Convolution 1x1
    conv1x1 = Conv1x1(inputs)

    # Loop through atrous rates
    for rate in atrous_rates:
        atrous_block = layers.Conv2D(filters, (3, 3), dilation_rate=rate, padding='same')(inputs)
        atrous_block = layers.BatchNormalization()(atrous_block)
        atrous_block = layers.Activation('relu')(atrous_block)
        atrous_block = layers.Dropout(dropout_rate)(atrous_block)
        atrous_blocks.append(atrous_block)

    # Concatenate atrous convolution results
    concatenated = layers.concatenate([conv1x1, atrous_blocks], axis=-1)
    
    # Global Average Pooling
    global_avg_pooling = layers.GlobalAveragePooling2D()(inputs)
    global_avg_pooling = tf.expand_dims(tf.expand_dims(global_avg_pooling, 1), 1)
    global_avg_pooling = layers.Conv2D(filters, (1, 1), padding='valid')(global_avg_pooling)
    global_avg_pooling = layers.BatchNormalization()(global_avg_pooling)
    global_avg_pooling = layers.Activation('relu')(global_avg_pooling)
    global_avg_pooling = layers.Dropout(dropout_rate)(global_avg_pooling)

    # Upsample global average pooling
    size = K.int_shape(inputs)[1:3]
    global_avg_pooling = layers.UpSampling2D(size=size, interpolation='bilinear')(global_avg_pooling)

    # Concatenate global average pooling with atrous convolution results
    concatenated = layers.concatenate([concatenated, global_avg_pooling], axis=-1)

    # 1x1 Convolution
    output = Conv1x1(concatenated)

    return output

def decodeur(inputs, backbone, input_layer, filters=256, upsampling_factor_aspp=4, dropout_rate=0.5, num_classes=21):
    # Upsampling by a factor 4
    aspp_output = layers.UpSampling2D(size=(upsampling_factor_aspp, upsampling_factor_aspp), interpolation='bilinear')(inputs)

    # Créer un modèle intermédiaire pour obtenir 'conv2_block3_out'
    intermediate_model = Model(inputs=backbone.input, outputs=backbone.get_layer('conv2_block3_out').output)

    # Récupérer les low_level_features
    low_level_features = intermediate_model(input_layer) 
    low_level_features = Conv1x1(low_level_features, filters=48)

    # Concatenate ASPP output with low level features
    concatenated = layers.concatenate([aspp_output, low_level_features], axis=-1)

    # Définir le nombre de couches de convolution
    num_conv_layers = 2

    # Utiliser une boucle for pour ajouter les couches de convolution
    for _ in range(num_conv_layers):
        concatenated = layers.Conv2D(filters, (3, 3), padding='same'))(concatenated)
        concatenated = layers.BatchNormalization()(concatenated)
        concatenated = layers.Activation('relu')(concatenated)
        concatenated = layers.Dropout(dropout_rate)(concatenated)

    concatenated = layers.UpSampling2D(size=(upsampling_factor_aspp, upsampling_factor_aspp), interpolation='bilinear')(concatenated)

    return concatenated


def deeplabv3_model(input_shape=(224, 224, 3), num_classes=21):
    # Input Layer
    input_layer = layers.Input(shape=input_shape, name='input_image')

    # Backbone (MobileNetV2 in this example)
    backbone = tf.keras.applications.ResNet101(input_shape=input_shape, include_top=False, weights='imagenet', pooling='avg')

    # Exclure les couches dense et de pooling global
    output_layer = backbone.get_layer('conv5_block3_out').output

    # Créer un nouveau modèle avec la sortie désirée
    resnet_model = Model(inputs=backbone.input, outputs=output_layer)

    # Fine-tuning des couches du backbone
    for layer in resnet_model.layers:
        layer.trainable = True
    
    multi_grid = (1, 2, 1)    
    # Nom des couches de convolution à remplacer dans conv5
    conv_layers_to_replace = ['conv5_block1_0_conv', 'conv5_block1_1_conv',
                            'conv5_block1_2_conv', 'conv5_block1_3_conv',
                            'conv5_block2_1_conv', 'conv5_block2_2_conv',
                            'conv5_block2_3_conv', 'conv5_block3_1_conv',
                            'conv5_block3_2_conv', 'conv5_block3_3_conv']

    # Modifier directement les couches de convolution dans conv5
    for layer in resnet_model.layers:
        if layer.name in conv_layers_to_replace and isinstance(layer, Conv2D):
            # Calculer le numéro du block en fonction du nom de la couche
            block_number = int(layer.name.split('_')[2][-1])  # Extrait le numéro du block de la couche
            # Configurer les taux de dilatation
            layer.strides = (1, 1)
            layer.dilation_rate = (2 * multi_grid[block_number - 1], 2 * multi_grid[block_number - 1])
    
            
    resnet_model_output = resnet_model(input_layer)
    
    # ASPP Block
    aspp_output = aspp(resnet_model_output, dropout_rate=0.8)

    # Decoder
    decoder_output = decodeur(aspp_output, backbone=backbone, input_layer=input_layer, dropout_rate=0.8)

    # Predictions
    predictions = layers.Conv2D(num_classes, (1, 1), padding='same', activation='softmax')(decoder_output)

    # Create model
    model = Model(inputs=input_layer, outputs=predictions, name='deeplabv3')

    return model

# Instantiate the model
model = deeplabv3_model()

plot_model(model, to_file='final_model.png')

# Afficher le résumé du modèle modifié
model.summary()

Model: "deeplabv3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 model_3 (Functional)           (None, 7, 7, 2048)   42658176    ['input_image[0][0]']            
                                                                                                  
 global_average_pooling2d_1 (Gl  (None, 2048)        0           ['model_3[0][0]']                
 obalAveragePooling2D)                                                                            
                                                                                          

In [ ]:
## Paramètres personnalisés pour l'entraînement
# Paramètres
batch_size = 16
epochs = 5

# Définition d'un taux d'apprentissage 'poly'
class PolyDecay(LearningRateSchedule):
    def __init__(self, initial_learning_rate, max_epochs, power=0.9):
        self.initial_learning_rate = initial_learning_rate
        self.max_epochs = max_epochs
        self.power = power

    def __call__(self, step):
        #return self.initial_learning_rate * (1 - step / float(self.max_epochs))**self.power
        return self.initial_learning_rate * (1 - step / self.max_epochs)**self.power

# Utilisation de la classe PolyDecay
initial_lr = 0.007  # Taux d'apprentissage initial
poly_decay = PolyDecay(initial_learning_rate=initial_lr, max_epochs=epochs)

# Création de l'optimiseur avec le taux d'apprentissage initial
optimizer = tf.keras.optimizers.Adam(learning_rate=poly_decay)


# Fonction de perte customisée
def custom_loss(y_true, y_pred):
    # Créer un masque pour exclure la classe du background (classe 0)
    mask = tf.math.not_equal(y_true, 0)

    # Calculez la perte de cross-entropy pour chaque position spatiale, en tenant compte du masque
    pixel_losses = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
    masked_pixel_losses = tf.boolean_mask(pixel_losses, mask)

    # Ajoutez une régularisation L2
    l2_loss = 0.01 * tf.reduce_sum([tf.nn.l2_loss(w) for w in model.trainable_variables])

    # Sommez les pertes masquées et la régularisation pour obtenir la perte totale
    total_loss = tf.reduce_sum(masked_pixel_losses) + l2_loss

    return total_loss


# Compilation du modèle
model.compile(optimizer=optimizer,
              loss=custom_loss,
              metrics=['accuracy'])

# Définition de l'arrêt anticipé (early stopping)
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
# Entraînement du modèle avec régularisation et early stopping
model.fit(train_dataset_combined.batch(batch_size), epochs=epochs, validation_data=val_dataset.batch(batch_size), callbacks=[early_stopping])